In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/12345k/IPL-Dataset/master/IPL/data.csv"

ipl_df = pd.read_csv(url)


print(ipl_df.head())

   id  season        city        date                  team1  \
0   1    2008   Bangalore  2008-04-18  Kolkata Knight Riders   
1   2    2008  Chandigarh  2008-04-19    Chennai Super Kings   
2   3    2008       Delhi  2008-04-19       Rajasthan Royals   
3   4    2008      Mumbai  2008-04-20         Mumbai Indians   
4   5    2008     Kolkata  2008-04-20        Deccan Chargers   

                         team2                  toss_winner toss_decision  \
0  Royal Challengers Bangalore  Royal Challengers Bangalore         field   
1              Kings XI Punjab          Chennai Super Kings           bat   
2             Delhi Daredevils             Rajasthan Royals           bat   
3  Royal Challengers Bangalore               Mumbai Indians           bat   
4        Kolkata Knight Riders              Deccan Chargers           bat   

   result  dl_applied                       winner  win_by_runs  \
0  normal           0        Kolkata Knight Riders          140   
1  normal         

In [2]:
ipl_df = ipl_df.dropna(subset=["winner"]).reset_index(drop=True)

print(ipl_df.head())
print("Shape after cleaning:", ipl_df.shape)

   id  season        city        date                  team1  \
0   1    2008   Bangalore  2008-04-18  Kolkata Knight Riders   
1   2    2008  Chandigarh  2008-04-19    Chennai Super Kings   
2   3    2008       Delhi  2008-04-19       Rajasthan Royals   
3   4    2008      Mumbai  2008-04-20         Mumbai Indians   
4   5    2008     Kolkata  2008-04-20        Deccan Chargers   

                         team2                  toss_winner toss_decision  \
0  Royal Challengers Bangalore  Royal Challengers Bangalore         field   
1              Kings XI Punjab          Chennai Super Kings           bat   
2             Delhi Daredevils             Rajasthan Royals           bat   
3  Royal Challengers Bangalore               Mumbai Indians           bat   
4        Kolkata Knight Riders              Deccan Chargers           bat   

   result  dl_applied                       winner  win_by_runs  \
0  normal           0        Kolkata Knight Riders          140   
1  normal         

In [3]:
conditions = [
    ipl_df["win_by_runs"] > 0,
    ipl_df["win_by_wickets"] > 0
]

choices = ["Runs", "Wickets"]

ipl_df["match_margin_type"] = np.select(
    conditions,
    choices,
    default="Tie/No Result"
)

print(
    ipl_df[
        ["team1", "team2", "winner",
         "win_by_runs", "win_by_wickets", "match_margin_type"]
    ].head(10)
)

                   team1                        team2  \
0  Kolkata Knight Riders  Royal Challengers Bangalore   
1    Chennai Super Kings              Kings XI Punjab   
2       Rajasthan Royals             Delhi Daredevils   
3         Mumbai Indians  Royal Challengers Bangalore   
4        Deccan Chargers        Kolkata Knight Riders   
5        Kings XI Punjab             Rajasthan Royals   
6        Deccan Chargers             Delhi Daredevils   
7    Chennai Super Kings               Mumbai Indians   
8        Deccan Chargers             Rajasthan Royals   
9        Kings XI Punjab               Mumbai Indians   

                        winner  win_by_runs  win_by_wickets match_margin_type  
0        Kolkata Knight Riders          140               0              Runs  
1          Chennai Super Kings           33               0              Runs  
2             Delhi Daredevils            0               9           Wickets  
3  Royal Challengers Bangalore            0         

In [4]:
team_cities = pd.DataFrame({
    "team": [
        "Mumbai Indians",
        "Chennai Super Kings",
        "Royal Challengers Bangalore",
        "Kolkata Knight Riders",
        "Sunrisers Hyderabad",
        "Rajasthan Royals",
        "Delhi Daredevils",
        "Kings XI Punjab",
        "Gujarat Lions",
        "Rising Pune Supergiant"
    ],
    "home_city": [
        "Mumbai",
        "Chennai",
        "Bengaluru",
        "Kolkata",
        "Hyderabad",
        "Jaipur",
        "Delhi",
        "Mohali",
        "Ahmedabad",
        "Pune"
    ]
})

# Merge city information for team1
ipl_df = ipl_df.merge(
    team_cities.rename(columns={
        "team": "team1",
        "home_city": "team1_home_city"
    }),
    on="team1",
    how="left"
)

# Merge city information for team2
ipl_df = ipl_df.merge(
    team_cities.rename(columns={
        "team": "team2",
        "home_city": "team2_home_city"
    }),
    on="team2",
    how="left"
)

print(
    ipl_df[
        ["team1", "team1_home_city",
         "team2", "team2_home_city"]
    ].head(10)
)

                   team1 team1_home_city                        team2  \
0  Kolkata Knight Riders         Kolkata  Royal Challengers Bangalore   
1    Chennai Super Kings         Chennai              Kings XI Punjab   
2       Rajasthan Royals          Jaipur             Delhi Daredevils   
3         Mumbai Indians          Mumbai  Royal Challengers Bangalore   
4        Deccan Chargers             NaN        Kolkata Knight Riders   
5        Kings XI Punjab          Mohali             Rajasthan Royals   
6        Deccan Chargers             NaN             Delhi Daredevils   
7    Chennai Super Kings         Chennai               Mumbai Indians   
8        Deccan Chargers             NaN             Rajasthan Royals   
9        Kings XI Punjab          Mohali               Mumbai Indians   

  team2_home_city  
0       Bengaluru  
1          Mohali  
2           Delhi  
3       Bengaluru  
4         Kolkata  
5          Jaipur  
6           Delhi  
7          Mumbai  
8          Jaipu

In [5]:
# Count matches won by each team in each season
wins = (
    ipl_df.groupby(["season", "winner"])
    .size()
    .reset_index(name="wins")
)

# Pivot: teams as rows and seasons as columns
pivot_wins = wins.pivot_table(
    index="winner",
    columns="season",
    values="wins",
    fill_value=0
)

print("Wins per team per season:")
print(pivot_wins)

# Top 3 team-season combinations with the most wins
top3 = wins.nlargest(3, "wins").reset_index(drop=True)

print("\nTop 3 teams with the most wins in a single season:")
print(top3)

Wins per team per season:
season                       2008  2009  2010  2011  2012  2013  2014  2015  \
winner                                                                        
Chennai Super Kings           9.0   8.0   9.0  11.0  10.0  12.0  10.0  10.0   
Deccan Chargers               2.0   9.0   8.0   6.0   4.0   0.0   0.0   0.0   
Delhi Daredevils              7.0  10.0   7.0   4.0  11.0   3.0   2.0   5.0   
Gujarat Lions                 0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   
Kings XI Punjab              10.0   7.0   4.0   7.0   8.0   8.0  12.0   3.0   
Kochi Tuskers Kerala          0.0   0.0   0.0   6.0   0.0   0.0   0.0   0.0   
Kolkata Knight Riders         6.0   3.0   7.0   8.0  12.0   6.0  11.0   7.0   
Mumbai Indians                7.0   5.0  11.0  10.0  10.0  13.0   7.0  10.0   
Pune Warriors                 0.0   0.0   0.0   4.0   4.0   4.0   0.0   0.0   
Rajasthan Royals             13.0   6.0   6.0   6.0   7.0  11.0   7.0   7.0   
Rising Pune Supergiants   